# Creacion Datasets Finales con metodo por Transecto y metodo General

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Preparación final: crea versiones 2D aplanadas (ml_2d) para algoritmos tipo árbol.
Solo se generan si todos los splits tienen al menos una muestra.
"""

import os
import json
import numpy as np
from pathlib import Path

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
WINDOWS_PARTITIONED_DIR = os.path.join(BASE_DIR, "windows_partitioned")

INPUT_DIR = WINDOWS_PARTITIONED_DIR
PATHS = {
    "by_transect_ml": os.path.join(INPUT_DIR, "by_transect", "ml"),
    "by_transect_dl": os.path.join(INPUT_DIR, "by_transect", "dl"),
    "global_ml": os.path.join(INPUT_DIR, "global", "ml"),
    "global_dl": os.path.join(INPUT_DIR, "global", "dl"),
}
WINDOW_IN = 72


def ensure_2d(X):
    if X.ndim == 3:
        n_samples, win_in, n_features = X.shape
        return X.reshape(n_samples, win_in * n_features)
    elif X.ndim == 2:
        return X
    else:
        raise ValueError(f"Dimensiones inesperadas: {X.ndim}")


def process_entity_ml(ml_dir, entity_name):
    entity_path = os.path.join(ml_dir, entity_name)
    if not os.path.isdir(entity_path):
        return False
    files = {
        'train_X': os.path.join(entity_path, "train_X.npy"),
        'val_X': os.path.join(entity_path, "val_X.npy"),
        'test_X': os.path.join(entity_path, "test_X.npy"),
        'train_y': os.path.join(entity_path, "train_y.npy"),
        'val_y': os.path.join(entity_path, "val_y.npy"),
        'test_y': os.path.join(entity_path, "test_y.npy"),
    }
    for name, path in files.items():
        if not os.path.exists(path):
            print(f"    Falta {name} para {entity_name}, se omite.")
            return False

    train_X = np.load(files['train_X'])
    val_X   = np.load(files['val_X'])
    test_X  = np.load(files['test_X'])
    train_y = np.load(files['train_y'])
    val_y   = np.load(files['val_y'])
    test_y  = np.load(files['test_y'])

    # Verificar que ningún split esté vacío
    if len(train_X) == 0 or len(val_X) == 0 or len(test_X) == 0:
        print(f"    Saltando {entity_name}: split vacío (train={len(train_X)}, val={len(val_X)}, test={len(test_X)})")
        return False

    train_X_2d = ensure_2d(train_X)
    val_X_2d   = ensure_2d(val_X)
    test_X_2d  = ensure_2d(test_X)

    out_dir = os.path.join(entity_path, "ml_2d")
    os.makedirs(out_dir, exist_ok=True)

    np.save(os.path.join(out_dir, "train_X.npy"), train_X_2d)
    np.save(os.path.join(out_dir, "val_X.npy"),   val_X_2d)
    np.save(os.path.join(out_dir, "test_X.npy"),  test_X_2d)
    np.save(os.path.join(out_dir, "train_y.npy"), train_y)
    np.save(os.path.join(out_dir, "val_y.npy"),   val_y)
    np.save(os.path.join(out_dir, "test_y.npy"),  test_y)

    print(f"    ML 2D generado para {entity_name}: {train_X_2d.shape}")
    return True


def process_ml_folder(ml_dir, description):
    if not os.path.exists(ml_dir):
        print(f"  No existe {ml_dir}")
        return
    entities = [d for d in os.listdir(ml_dir) if os.path.isdir(os.path.join(ml_dir, d))]
    if not entities:
        print(f"  No hay entidades en {ml_dir}")
        return
    print(f"\n--- Procesando {description} ---")
    for entity in sorted(entities):
        process_entity_ml(ml_dir, entity)


def generate_metadata():
    metadata = {
        "by_transect": {"ml": {}, "dl": {}},
        "global": {"ml": {}, "dl": {}}
    }
    def get_shapes(base_dir, subdir, entity):
        shapes = {}
        for split in ['train', 'val', 'test']:
            X_path = os.path.join(base_dir, subdir, entity, f"{split}_X.npy")
            y_path = os.path.join(base_dir, subdir, entity, f"{split}_y.npy")
            if os.path.exists(X_path):
                X = np.load(X_path, mmap_mode='r')
                shapes[f"{split}_X"] = list(X.shape)
            if os.path.exists(y_path):
                y = np.load(y_path, mmap_mode='r')
                shapes[f"{split}_y"] = list(y.shape)
        return shapes
    # by_transect
    for sub in ['ml', 'dl']:
        base = PATHS[f"by_transect_{sub}"]
        if os.path.exists(base):
            for entity in os.listdir(base):
                if os.path.isdir(os.path.join(base, entity)):
                    metadata["by_transect"][sub][entity] = get_shapes(PATHS["by_transect_ml"] if sub=='ml' else PATHS["by_transect_dl"], sub, entity)
                    if sub == 'ml':
                        ml_2d_path = os.path.join(base, entity, "ml_2d")
                        if os.path.exists(ml_2d_path):
                            metadata["by_transect"]["ml"][entity]["ml_2d"] = get_shapes(base, entity, "ml_2d")
    # global
    for sub in ['ml', 'dl']:
        base = PATHS[f"global_{sub}"]
        if os.path.exists(base):
            for entity in os.listdir(base):
                if os.path.isdir(os.path.join(base, entity)):
                    metadata["global"][sub][entity] = get_shapes(PATHS["global_ml"] if sub=='ml' else PATHS["global_dl"], sub, entity)
                    if sub == 'ml':
                        ml_2d_path = os.path.join(base, entity, "ml_2d")
                        if os.path.exists(ml_2d_path):
                            metadata["global"]["ml"][entity]["ml_2d"] = get_shapes(base, entity, "ml_2d")
    meta_path = os.path.join(INPUT_DIR, "dataset_metadata.json")
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"\nMetadatos guardados en {meta_path}")


if __name__ == "__main__":
    print("Preparación final de datasets (ML 2D)")
    process_ml_folder(PATHS["by_transect_ml"], "datos por transecto (ML)")
    process_ml_folder(PATHS["global_ml"], "datos globales (ML)")
    generate_metadata()
    print("Proceso completado.")

Preparación final de datasets (ML 2D)

--- Procesando datos por transecto (ML) ---
    ML 2D generado para Transecto_1: (8698, 1728)
    ML 2D generado para Transecto_2: (8399, 1728)

--- Procesando datos globales (ML) ---
    ML 2D generado para T1_E1_Alicante: (8604, 1656)
    ML 2D generado para T2_E1_Elche: (8104, 1656)

Metadatos guardados en /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/windows_partitioned/dataset_metadata.json
Proceso completado.
